# 01. Audit Kualitas Data & Pembersihan: Sample Superstore

**Proyek**: Analisis Kinerja Penjualan & Profitabilitas Retail  
**Tujuan**: Melakukan audit integritas data, profiling komprehensif, validasi aturan bisnis, dan menyiapkan dataset terproses yang bersih untuk pemodelan analitik SQL, eksplorasi EDA, dan dashboard interaktif Streamlit.

In [8]:
import pandas as pd
import numpy as np
import sqlite3
import os

# Konfigurasi tampilan data
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

## 1. Pemuatan Data & Pemeriksaan Awal (Data Ingestion & Initial Inspection)
Memuat dataset yang telah diproses untuk memverifikasi struktur tabel, jumlah baris transaksi, dan kesesuaian tipe data tiap kolom.

In [3]:
data_path = '../data/processed/Sample_Superstore_Cleaned_Final.csv'
df = pd.read_csv(data_path)
print(f"Total catatan transaksi: {len(df):,}")
print(f"Total kolom/fitur: {df.shape[1]}")
df.info()

Total catatan transaksi: 9,994
Total kolom/fitur: 21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         9994 non-null   int64  
 1   order_id       9994 non-null   object 
 2   order_date     9994 non-null   object 
 3   ship_date      9994 non-null   object 
 4   ship_mode      9994 non-null   object 
 5   customer_id    9994 non-null   object 
 6   customer_name  9994 non-null   object 
 7   segment        9994 non-null   object 
 8   country        9994 non-null   object 
 9   city           9994 non-null   object 
 10  state          9994 non-null   object 
 11  postal_code    9994 non-null   int64  
 12  region         9994 non-null   object 
 13  product_id     9994 non-null   object 
 14  category       9994 non-null   object 
 15  sub_category   9994 non-null   object 
 16  product_name   9994 non-null   object 
 17 

> 📌 **Highlight Proses & Temuan Kunci #1:**
> - **Volume Data:** Dataset memuat **9.994 baris transaksi** dengan **21 kolom** atribut pelanggan, produk, geografi, dan keuangan.
> - **Granularitas Data:** Setiap baris mewakili 1 item produk dalam suatu transaksi pemesanan (*line-item grain*).
> - **Kelayakan Skema:** Struktur data lengkap dan siap digunakan untuk proses analisis tanpa perlu pemotongan baris.

## 2. Audit Nilai Hilang & Kelengkapan Data (Missing Value Audit)
Memverifikasi bahwa dimensi transaksional penting dan metrik keuangan tidak memiliki nilai kosong (null/NaN).

In [4]:
null_summary = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase Missing (%)': (df.isnull().sum() / len(df)) * 100
})
print("Hasil Audit Missing Values:")
if null_summary['Jumlah Missing'].sum() == 0:
    print("Sempurna: Tidak ditemukan nilai kosong (0.00% missing values). Dataset 100% lengkap!")
else:
    display(null_summary[null_summary['Jumlah Missing'] > 0])

Hasil Audit Missing Values:
Sempurna: Tidak ditemukan nilai kosong (0.00% missing values). Dataset 100% lengkap!


> 📌 **Highlight Proses & Temuan Kunci #2:**
> - **Kelengkapan 100%:** Seluruh 21 kolom memiliki kelengkapan data penuh tanpa ada nilai yang hilang (`0 missing values`).
> - **Kualitas Data:** Tidak diperlukan imputasi buatan atau penghapusan baris, sehingga menjaga keaslian seluruh catatan riwayat transaksi.

## 3. Deteksi Duplikasi Data (Duplicate Detection)
Memeriksa keunikan kunci primer (`row_id`) serta memastikan tidak adanya baris transaksi yang terduplikasi secara identik.

In [5]:
duplicate_row_id = df['row_id'].duplicated().sum()
print(f"Jumlah duplikasi row_id: {duplicate_row_id}")

duplicate_full = df.duplicated().sum()
print(f"Jumlah baris duplikat penuh: {duplicate_full}")

Jumlah duplikasi row_id: 0
Jumlah baris duplikat penuh: 0


> 📌 **Highlight Proses & Temuan Kunci #3:**
> - **Keunikan Primary Key:** Kolom `row_id` 100% unik (`0 duplikat`), menjamin perannya sebagai kunci primer yang valid.
> - **Bebas Anomali Ganda:** Tidak ada pencatatan transaksi ganda pada sistem data (*zero identical row duplicates*).

## 4. Validasi Aturan Bisnis & Batasan Numerik (Business Rule & Boundary Validation)
Memvalidasi 4 aturan bisnis utama:
1. Urutan kronologis: Tanggal kirim (`ship_date`) harus `>=` Tanggal pesan (`order_date`).
2. Nilai penjualan (`sales`) dan kuantitas unit (`quantity`) harus bernilai positif.
3. Tingkat diskon (`discount`) harus berada pada rentang logis 0% hingga 100% (0.0 hingga 1.0).
4. Meninjau rentang nilai laba/rugi (`profit`).

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Pemeriksaan kronologi tanggal pengiriman
invalid_dates = (df['ship_date'] < df['order_date']).sum()
print(f"Jumlah tanggal pengiriman tidak valid (ship_date < order_date): {invalid_dates}")

# Ringkasan statistik metrik numerik
print("\nRingkasan Statistik Angka Kunci:")
display(df[['sales', 'quantity', 'discount', 'profit']].describe())

Jumlah tanggal pengiriman tidak valid (ship_date < order_date): 0

Ringkasan Statistik Angka Kunci:


,sales,quantity,discount,profit
count,9994.0000,9994.0000,9994.0000,9994.0000
mean,229.8580,3.7896,0.1562,28.6569
std,623.2451,2.2251,0.2065,234.2601
min,0.4440,1.0000,0.0000,-6599.9780
25%,17.2800,2.0000,0.0000,1.7288
50%,54.4900,3.0000,0.2000,8.6665
75%,209.9400,5.0000,0.2000,29.3640
max,22638.4800,14.0000,0.8000,8399.9760


> 📌 **Highlight Proses & Temuan Kunci #4:**
> - **Kronologi Valid:** 100% baris pesanan memiliki `ship_date >= order_date`. Tidak ada tanggal kirim mundur.
> - **Kepatuhan Batasan:** Nilai `sales` minimum $0,44 (selalu > 0), `quantity` minimal 1 unit, dan `discount` maksimum adalah 80% (0,80).
> - **Indikasi Masalah Finansial:** Kolom `profit` memiliki nilai minimum **-$6.599,98**, yang membuktikan bahwa sebagian transaksi ritel mengalami kerugian sangat dalam akibat diskon berlebih.

## 5. Pipeline Ekspor Database SQLite (`superstore.db`)
Menyimpan dataset yang telah terverifikasi ke dalam tabel `orders` di database SQLite `superstore.db` untuk keperluan eksekusi kueri analitik SQL.

In [7]:
db_path = '../data/processed/superstore.db'
conn = sqlite3.connect(db_path)
df.to_sql('orders', conn, if_exists='replace', index=False)

# Verifikasi kueri ke database SQLite
test_query = """SELECT 
    COUNT(*) AS row_count, 
    ROUND(SUM(sales), 2) AS total_sales, 
    ROUND(SUM(profit), 2) AS total_profit
FROM orders;"""
res = pd.read_sql(test_query, conn)
print("Verifikasi Database SQLite Berhasil:")
display(res)
conn.close()

Verifikasi Database SQLite Berhasil:


,row_count,total_sales,total_profit
0,9994,2297200.8600,286397.0200


> 📌 **Highlight Proses & Temuan Kunci #5:**
> - **Persistensi Database:** Tabel `orders` berhasil terisi dan terindeks di `superstore.db` tanpa kehilangan baris data (persis 9.994 baris).
> - **Integritas Nilai:** Total pendapatan **$2.297.200,86** dan total laba **$286.397,02** di SQLite 100% konsisten dengan data Pandas.

## 6. Laporan Ringkasan Kualitas Data (Summary Audit Report)
| Komponen Pemeriksaan | Hasil Audit | Penjelasan Status |
| :--- | :--- | :--- |
| **Jumlah Baris Data** | 9.994 baris | Utuh 100%, tidak ada baris yang hilang atau terpotong |
| **Missing Values** | 0 (0,00%) | Bersih di seluruh 21 kolom atribut |
| **Duplikasi Data** | 0 baris | `row_id` berfungsi sebagai kunci primer yang unik |
| **Urutan Kronologis** | Lolos Validasi | Seluruh `ship_date >= order_date` |
| **Batasan Angka Bisnis** | Lolos Validasi | `sales > 0`, `quantity >= 1`, `0.0 <= discount <= 0.8` |
| **Kesiapan Database** | Siap Pakai | Telah diimpor ke SQLite `superstore.db` tabel `orders` |